# 04 — Transfer Learning avec ResNet18

On remplace notre CNN from scratch par **ResNet18** pré-entraîné sur ImageNet.

**Objectif :** dépasser les scores de la baseline :
- Accuracy baseline : 86%
- Recall PNEUMONIA baseline : 92%
- FN baseline : 30 pneumonies manquées

**Stratégie en 2 phases :**
1. **Feature extraction** : on gèle tous les poids de ResNet, on entraîne seulement la dernière couche
2. **Fine-tuning** : on dégèle les dernières couches et on affine avec un très petit learning rate

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH = Path('/content/chest_xray')
print(f'Device : {device}')

# Transforms identiques au notebook 03
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10, fill=128),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
test_tf = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_ds = datasets.ImageFolder(DATA_PATH / 'train', transform=train_tf)
val_ds   = datasets.ImageFolder(DATA_PATH / 'val',   transform=test_tf)
test_ds  = datasets.ImageFolder(DATA_PATH / 'test',  transform=test_tf)
train_ld = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_ld   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_ld  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

# Class weights
labels_list = [l for _, l in train_ds.samples]
cw = torch.tensor([len(labels_list) / (2 * labels_list.count(i)) for i in [0, 1]]).to(device)
print(f'Class weights : NORMAL={cw[0]:.3f} | PNEUMONIA={cw[1]:.3f}')
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## 2. Charger ResNet18 pré-entraîné

ResNet18 a été entraîné sur **ImageNet** : 1,2 million d'images, 1000 classes.
Il sait déjà détecter des centaines de patterns visuels.

On va :
1. Télécharger ResNet18 avec ses poids ImageNet
2. **Geler** tous ses poids (on ne les modifie pas)
3. Remplacer sa dernière couche (1000 classes → 2 classes)
4. Entraîner **uniquement** cette nouvelle couche

**Pourquoi geler d'abord ?**
Si on entraîne tout le réseau d'un coup, les poids pré-entraînés risquent d'être
détruits par un learning rate trop élevé. On procède en 2 phases pour être plus stable.

In [ ]:
# Charger ResNet18 pré-entraîné
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Phase 1 : geler TOUS les poids
for param in model.parameters():
    param.requires_grad = False

# Remplacer la dernière couche FC (512 → 2 classes)
# model.fc est la couche finale de ResNet18
num_features = model.fc.in_features  # = 512
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 2)
)
# Les nouvelles couches ont requires_grad=True par défaut

model = model.to(device)

# Compter les paramètres entraînables vs gelés
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Paramètres total      : {total:,}')
print(f'Paramètres entraînables : {trainable:,} ({100*trainable/total:.1f}%)')
print(f'Paramètres gelés      : {total-trainable:,} ({100*(total-trainable)/total:.1f}%)')

## 3. Phase 1 — Feature Extraction (5 époques)

On entraîne uniquement la nouvelle tête de classification.
ResNet sert de **extracteur de features** figé.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=cw)
# On optimise SEULEMENT les paramètres qui ont requires_grad=True
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

def run(model, loader, train=True):
    model.train() if train else model.eval()
    loss_sum, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labs in loader:
            imgs, labs = imgs.to(device), labs.to(device)
            if train: optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labs)
            if train: loss.backward(); optimizer.step()
            loss_sum += loss.item()
            correct  += (out.argmax(1) == labs).sum().item()
            total    += labs.size(0)
    return loss_sum / len(loader), correct / total

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('PHASE 1 — Feature Extraction (poids ResNet gelés)')
print('='*65)
for ep in range(5):
    tl, ta = run(model, train_ld, train=True)
    vl, va = run(model, val_ld,   train=False)
    scheduler.step(vl)
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    saved = ''
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), '/content/resnet18_best.pt')
        saved = '✓'
    print(f'Époque {ep+1:>2}/5 | Train loss:{tl:.4f} acc:{ta:.3f} | Val loss:{vl:.4f} acc:{va:.3f} {saved}')
print('='*65)

## 4. Phase 2 — Fine-tuning (10 époques)

On **dégèle** les dernières couches de ResNet pour les adapter à nos radios.

**Pourquoi seulement les dernières couches ?**
- Les premières couches détectent des contours/textures basiques → universels, inutile de les modifier
- Les dernières couches détectent des patterns de haut niveau → spécifiques à ImageNet, on les adapte à nos radios

**Pourquoi un learning rate 10x plus petit ?**
Les poids pré-entraînés sont déjà bons — on veut les affiner doucement, pas les détruire.

In [ ]:
# Dégeler le dernier bloc de ResNet (layer4) + la fc
for param in model.layer4.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Paramètres entraînables après dégel : {trainable:,} ({100*trainable/total:.1f}%)')

# Learning rate 10x plus petit pour le fine-tuning
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

print('\nPHASE 2 — Fine-tuning (layer4 + fc dégelés)')
print('='*65)
for ep in range(10):
    tl, ta = run(model, train_ld, train=True)
    vl, va = run(model, val_ld,   train=False)
    scheduler.step(vl)
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    saved = ''
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), '/content/resnet18_best.pt')
        saved = '✓'
    print(f'Époque {ep+6:>2}/15 | Train loss:{tl:.4f} acc:{ta:.3f} | Val loss:{vl:.4f} acc:{va:.3f} {saved}')
print('='*65)
print('Entraînement terminé !')

## 5. Courbes d'apprentissage

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ['loss', 'acc'], ['Loss', 'Accuracy']):
    ax.plot(epochs, history[f'train_{metric}'], 'b-o', label='Train')
    ax.plot(epochs, history[f'val_{metric}'],   'r-o', label='Validation')
    ax.axvline(x=5.5, color='gray', linestyle='--', label='Début fine-tuning')
    ax.set_title(f'Courbe de {title}', fontsize=13)
    ax.set_xlabel('Époque')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('ResNet18 — Feature Extraction + Fine-tuning', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Évaluation finale + Comparaison avec la baseline

In [ ]:
model.load_state_dict(torch.load('/content/resnet18_best.pt', map_location=device))
model.eval()

preds, labs_all = [], []
with torch.no_grad():
    for imgs, labs in test_ld:
        preds.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
        labs_all.extend(labs.tolist())

print('=== Résultats ResNet18 sur le Test Set ===')
print(classification_report(labs_all, preds, target_names=['NORMAL', 'PNEUMONIA']))

cm = confusion_matrix(labs_all, preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['NORMAL', 'PNEUMONIA'],
            yticklabels=['NORMAL', 'PNEUMONIA'], ax=ax)
ax.set_xlabel('Prédiction'); ax.set_ylabel('Réalité')
ax.set_title('Matrice de confusion — ResNet18')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TP={tp} | TN={tn} | FP={fp} | FN={fn} ⚠️')

# Comparaison
print('\n=== Comparaison Baseline CNN vs ResNet18 ===')
print(f'{"Métrique":<25} {"Baseline CNN":>15} {"ResNet18":>12}')
print('-'*55)
baseline = {'accuracy': 0.86, 'recall_pneumonia': 0.92, 'fn': 30}
resnet   = {
    'accuracy': (tp+tn)/(tp+tn+fp+fn),
    'recall_pneumonia': tp/(tp+fn),
    'fn': fn
}
print(f'{"Accuracy":<25} {baseline["accuracy"]:>15.2f} {resnet["accuracy"]:>12.2f}')
print(f'{"Recall PNEUMONIA":<25} {baseline["recall_pneumonia"]:>15.2f} {resnet["recall_pneumonia"]:>12.2f}')
print(f'{"Faux Négatifs (FN)":<25} {baseline["fn"]:>15} {resnet["fn"]:>12}')

## 7. Conclusions

Note ici tes observations :
- Le transfer learning a-t-il amélioré les scores ?
- Combien de pneumonies manquées en moins ?
- Voit-on de l'overfitting ?
- Quelle phase a le plus contribué (feature extraction ou fine-tuning) ?

---
**Prochain notebook :** `05_evaluation.ipynb`
Analyse approfondie des erreurs + visualisation Grad-CAM
(ce que le modèle regarde vraiment dans les radios)